# 📊 Python Complexity Analysis — The Reference Guide
### *Big O, Amortized, Master Theorem, and Built-in Complexities*

---

> **Mental Model First:**
> Complexity is a speedometer, not a stopwatch. It doesn't tell you how fast
> your code runs on *this* machine — it tells you how the *work grows* as input
> doubles, triples, or goes to infinity. Two machines, same speedometer reading,
> same growth shape.

---

## 📋 Table of Contents

| # | Section |
|---|---------|
| 1 | [Growth Classes — The Big Picture](#1) |
| 2 | [Python Built-in Complexities](#2) |
| 3 | [Amortized Analysis](#3) |
| 4 | [Recurrence Relations & Master Theorem](#4) |
| 5 | [Space Complexity Patterns](#5) |
| 6 | [Decision Map — Reading Interview Problems](#6) |
| 7 | [Cheat Sheet](#7) |


<a id='1'></a>

## 1. Growth Classes — The Big Picture

---

```
GROWTH CLASSES (input n=1000, rough op counts)

  CLASS       EXAMPLE n=1000      GROWTH PATTERN
  ──────────────────────────────────────────────────────────────
  O(1)        1 op                Flat line. Size doesn't matter.
  O(log n)    ~10 ops             Doubles input → +1 step
  O(n)        1,000 ops           Linear. One pass.
  O(n log n)  ~10,000 ops         Sorting sweet spot.
  O(n²)       1,000,000 ops       Nested loops. Hurts at n>10k.
  O(n³)       10^9 ops            Triple nested. Only n<500.
  O(2^n)      10^301 ops          Exponential. Only n<30.
  O(n!)       Astronomical        Only n<12.

  VISUALIZATION (relative ops, log scale)

  O(1)      ─────────────────────────  (flat)
  O(log n)  ──╱──────────────────────  (gentle curve)
  O(n)      ────╱─────────────────────  (diagonal)
  O(n log n) ────╱╲────────────────────  (slightly steeper)
  O(n²)     ──────────╱───────────────  (parabola)
  O(2^n)    ─────────────────────────╱  (wall)

  RULE OF THUMB: n=10^8 ops in ~1 second
    n ≤ 10^8  →  O(n) or better is safe
    n ≤ 10^6  →  O(n log n) is safe
    n ≤ 10^4  →  O(n²) is safe
    n ≤ 500   →  O(n³) might pass
    n ≤ 20    →  O(2^n) can work
    n ≤ 12    →  O(n!) can work
```


<a id='2'></a>

## 2. Python Built-in Complexities

---

```
LIST (dynamic array)
  Operation          Average    Worst     Notes
  ─────────────────────────────────────────────────────────
  list[i]            O(1)       O(1)      index access
  list.append(x)     O(1)*      O(n)      *amortized; realloc rare
  list.pop()         O(1)       O(1)      pop from end
  list.pop(0)        O(n)       O(n)      shift everything left
  list.insert(i,x)   O(n)       O(n)      shift right of i
  list.remove(x)     O(n)       O(n)      search + shift
  x in list          O(n)       O(n)      linear scan
  len(list)          O(1)       O(1)      stored attribute
  list.sort()        O(n log n) O(n log n) TimSort
  list[a:b]          O(b-a)     O(n)      slice = copy

DICT (hash table)
  Operation          Average    Worst     Notes
  ─────────────────────────────────────────────────────────
  d[k]               O(1)       O(n)      worst = all hash collide
  d[k] = v           O(1)       O(n)
  del d[k]           O(1)       O(n)
  k in d             O(1)       O(n)
  len(d)             O(1)

SET (hash table, same as dict)
  x in s             O(1)       O(n)
  s.add(x)           O(1)       O(n)
  s.remove(x)        O(1)       O(n)
  s | t (union)      O(len(s)+len(t))
  s & t (intersect)  O(min(len(s),len(t)))

DEQUE (doubly-linked list of blocks)
  appendleft/left    O(1)       O(1)      both ends O(1)
  popleft/right      O(1)       O(1)
  deque[i]           O(n)       O(n)      NO O(1) random access!

HEAPQ (binary heap in a list)
  heappush           O(log n)
  heappop            O(log n)
  heapify            O(n)       NOT O(n log n)!
  heapreplace        O(log n)   push + pop in one op

STRING
  s + t              O(len(s)+len(t))  creates new string
  s[i]               O(1)
  x in s             O(n*m)    substring search
  s.join(lst)        O(total)  use join, not +=
```


In [ ]:
# Live demonstrations of complexity surprises
import time, collections, heapq

# ── List: O(1) append vs O(n) insert(0) ──────────────────────────────────────
N = 10_000

start = time.perf_counter()
lst = []
for i in range(N):
    lst.append(i)        # O(1) amortized — each item ~1 op
t_append = time.perf_counter() - start

start = time.perf_counter()
lst2 = []
for i in range(N):
    lst2.insert(0, i)    # O(n) — shifts all existing elements right
t_insert = time.perf_counter() - start

print(f"append x{N}: {t_append*1000:.2f}ms")
print(f"insert(0) x{N}: {t_insert*1000:.2f}ms")
print(f"insert(0) is ~{t_insert/t_append:.0f}x slower — O(n) vs O(1)")

# ── heapify is O(n), not O(n log n) ──────────────────────────────────────────
import random
data = list(range(N))
random.shuffle(data)

start = time.perf_counter()
heapq.heapify(data[:])   # O(n) — sifts from bottom up; each level less work
t_heapify = time.perf_counter() - start

start = time.perf_counter()
h = []
for x in data:
    heapq.heappush(h, x) # O(n log n) — n pushes, each O(log n)
t_push = time.perf_counter() - start

print(f"heapify O(n): {t_heapify*1000:.3f}ms")
print(f"n pushes O(n log n): {t_push*1000:.3f}ms")
print(f"heapify is ~{t_push/t_heapify:.0f}x faster")

# ── String concatenation: join vs += ─────────────────────────────────────────
words = ["word"] * 1000

start = time.perf_counter()
result = ""
for w in words:
    result += w          # O(n²) total — each += creates a new string
t_plus = time.perf_counter() - start

start = time.perf_counter()
result2 = "".join(words) # O(n) total — one allocation, one copy
t_join = time.perf_counter() - start

print(f"str += x1000: {t_plus*1000:.3f}ms")
print(f"join x1000:   {t_join*1000:.3f}ms")
print("Complexity demo complete.")


<a id='3'></a>

## 3. Amortized Analysis

---

```
WHAT IS AMORTIZED?
  A single op might be slow occasionally, but the AVERAGE over many ops is fast.
  You "pre-pay" for future cheap ops during an expensive one.

CLASSIC EXAMPLE: Python list.append()
  ─────────────────────────────────────────────────────
  Capacity:  1  →  2  →  4  →  8  →  16
  Cost:      1     2     4     8      16   (copy old)
             1     1     1     1       1   (normal appends)

  Total cost for n appends: ~2n ops (copy ops + append ops)
  Amortized per-append: O(2n / n) = O(1)

  WHY: When you double capacity, you just did n cheap appends
       to afford the next copy. The copy cost is "spread" over those n ops.

OTHER AMORTIZED O(1) STRUCTURES:
  - collections.deque: appendleft/popleft — doubly-linked blocks
  - dict/set: insert with occasional rehash
  - Union-Find with path compression: nearly O(1) per find

AMORTIZED ≠ WORST CASE:
  ❌ "O(1) means every single call is O(1)"
  ✅ "O(1) amortized means the average over all calls is O(1)"

WHEN TO WORRY:
  Real-time systems (robotics, game loops) where any single
  op latency matters — use pre-allocated structures instead.
```


In [ ]:
# Visualize list doubling — see the copy spikes
import sys

lst = []
prev_cap = 0
print(f"{'n':>6}  {'cap':>8}  {'growth event':>15}")
print("-" * 40)

for i in range(33):
    lst.append(i)
    # sys.getsizeof gives bytes; rough capacity from size
    # List overhead: 56 bytes base + 8 bytes per slot
    size_bytes = sys.getsizeof(lst)
    cap = (size_bytes - 56) // 8   # approximate capacity in slots
    if cap != prev_cap:
        marker = f"<-- doubled to {cap}" if cap > 1 else ""
        print(f"{i+1:>6}  {cap:>8}  {marker}")
        prev_cap = cap

# ── Amortized stack push/pop ──────────────────────────────────────────────────
# Demonstrate that even though pop occasionally triggers work,
# the amortized cost stays O(1) over a long sequence.

class AmortizedStack:
    # Two-stack queue trick: push to inbox, pop from outbox
    # When outbox empty, move all from inbox — O(n) once per element
    def __init__(self):
        self.inbox = []    # new items land here — cheap push
        self.outbox = []   # items leave from here — cheap pop

    def push(self, x):
        self.inbox.append(x)   # always O(1)

    def pop(self):
        if not self.outbox:
            # Move all: each item crosses once in its lifetime → amortized O(1)
            while self.inbox:
                self.outbox.append(self.inbox.pop())
        return self.outbox.pop()

q = AmortizedStack()
for i in range(6):
    q.push(i)
    print(f"push({i})  inbox={q.inbox}  outbox={q.outbox}")

print()
for _ in range(6):
    val = q.pop()
    print(f"pop() → {val}  inbox={q.inbox}  outbox={q.outbox}")

print("Amortized demo complete.")


<a id='4'></a>

## 4. Recurrence Relations & Master Theorem

---

```
RECURRENCE RELATION
  T(n) = a·T(n/b) + f(n)
  └─ a = number of subproblems
     b = factor by which input shrinks
     f(n) = work done outside recursive calls

MASTER THEOREM (3 cases)
  Compare f(n) with n^(log_b(a)):

  CASE 1: f(n) = O(n^(log_b(a) - ε))   [recursive work dominates]
          → T(n) = Θ(n^log_b(a))

  CASE 2: f(n) = Θ(n^(log_b(a)))        [equal work at every level]
          → T(n) = Θ(n^log_b(a) · log n)

  CASE 3: f(n) = Ω(n^(log_b(a) + ε))   [non-recursive work dominates]
          → T(n) = Θ(f(n))

COMMON EXAMPLES
  ──────────────────────────────────────────────────────────────
  Algorithm          Recurrence         log_b(a)  Result
  ──────────────────────────────────────────────────────────────
  Binary search      T(n)=T(n/2)+O(1)   log2(1)=0  O(log n) [Case 2]
  Merge sort         T(n)=2T(n/2)+O(n)  log2(2)=1  O(n log n) [Case 2]
  Quick sort (avg)   T(n)=2T(n/2)+O(n)  log2(2)=1  O(n log n) [Case 2]
  Quick sort (worst) T(n)=T(n-1)+O(n)   pivot at end  O(n²) [no master thm]
  Strassen matrix    T(n)=7T(n/2)+O(n²) log2(7)≈2.81 O(n^2.81) [Case 1]
  Tree height        T(n)=2T(n/2)+O(1)  log2(2)=1  O(n) [Case 1]
  ──────────────────────────────────────────────────────────────

QUICK MENTAL MODEL:
  "Divide and combine" → usually O(n log n)
  "Divide, one branch" → usually O(log n)
  "All pairs"          → O(n²)
```


In [ ]:
# Verify master theorem predictions with timing
import time, random

def merge_sort(arr):
    # T(n) = 2T(n/2) + O(n)  → Master theorem Case 2 → O(n log n)
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])   # a=2 subproblems
    right = merge_sort(arr[mid:])  # each of size n/2  → b=2
    return merge(left, right)      # merge = O(n)  → f(n)=O(n)

def merge(left, right):
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i]); i += 1
        else:
            result.append(right[j]); j += 1
    return result + left[i:] + right[j:]

def binary_search(arr, target):
    # T(n) = T(n/2) + O(1)  → Master theorem Case 2 → O(log n)
    lo, hi = 0, len(arr) - 1
    while lo <= hi:                # O(1) work per level
        mid = (lo + hi) // 2      # one subproblem of size n/2  → a=1, b=2
        if arr[mid] == target: return mid
        elif arr[mid] < target: lo = mid + 1
        else: hi = mid - 1
    return -1

# Time both across different input sizes
print(f"{'n':>8}  {'merge_sort(ms)':>16}  {'n*log2(n) ratio':>16}")
import math
for n in [1000, 2000, 4000, 8000]:
    data = list(range(n)); random.shuffle(data)
    start = time.perf_counter()
    for _ in range(10): merge_sort(data[:])
    t = (time.perf_counter() - start) / 10 * 1000
    expected_ratio = n * math.log2(n)
    print(f"{n:>8}  {t:>16.3f}  ~O(n log n) [n·log2(n)={expected_ratio:.0f}]")

print()
arr = list(range(10_000))
print(f"binary_search(arr, 9999) = {binary_search(arr, 9999)}")
print(f"binary_search(arr, -1)   = {binary_search(arr, -1)}")
print("Master theorem demo complete.")


<a id='5'></a>

## 5. Space Complexity Patterns

---

```
SPACE COMPLEXITY = EXTRA memory your algorithm uses
  (Usually excludes the input itself unless it's modified in-place)

COMMON PATTERNS
  ──────────────────────────────────────────────────────────────
  Pattern                     Space         Why
  ──────────────────────────────────────────────────────────────
  Single variables            O(1)          No growth with n
  Single pass with one dict   O(n)          Dict grows with input
  Recursive function          O(depth)      Each frame on call stack
  Merge sort                  O(n)          Temp arrays at each level
  Quick sort (in-place)       O(log n)      Call stack depth only
  BFS                         O(w)          w = max queue width
  DFS                         O(h)          h = max stack/recursion depth
  DP 2D table                 O(n·m)        Full grid stored
  DP with rolling array       O(m)          Only 2 rows at a time
  ──────────────────────────────────────────────────────────────

CALL STACK SPACE:
  def f(n):       # Each call = one stack frame
      return f(n-1)  # Depth n → O(n) space
                     # Python default limit: 1000 frames

  # Tail recursive: same O(n) in Python (no TCO)
  # Use iterative to get O(1) stack space

SPACE OPTIMIZATION TRICKS:
  ❌  dp = [[0]*m for _ in range(n)]   # O(n·m)
  ✅  prev = [0]*m; curr = [0]*m       # O(m) rolling rows

  ❌  result = []
      for x in data: result.append(transform(x))   # O(n)
  ✅  return (transform(x) for x in data)           # O(1) generator

  In-place reversal, two-pointer swaps: O(1) extra space
```


In [ ]:
import sys

# ── Demonstrate call stack depth (O(depth) space) ────────────────────────────
def recursive_sum(n):
    # Each call adds one frame — O(n) stack space
    if n == 0:
        return 0
    return n + recursive_sum(n - 1)   # stack depth = n

def iterative_sum(n):
    # O(1) space — only one frame regardless of n
    total = 0
    while n > 0:
        total += n
        n -= 1
    return total

print(f"recursive_sum(100) = {recursive_sum(100)}")
print(f"iterative_sum(100) = {iterative_sum(100)}")

# ── DP space optimization: rolling array ─────────────────────────────────────
def count_paths_full(m, n):
    # O(m·n) space — stores entire grid
    dp = [[1]*n for _ in range(m)]
    for i in range(1, m):
        for j in range(1, n):
            dp[i][j] = dp[i-1][j] + dp[i][j-1]
    return dp[m-1][n-1]

def count_paths_rolling(m, n):
    # O(n) space — only keep previous row
    prev = [1] * n
    for i in range(1, m):
        curr = [1] * n         # first column always 1
        for j in range(1, n):
            curr[j] = prev[j] + curr[j-1]  # above + left
        prev = curr            # discard old row
    return prev[n-1]

m, n = 5, 5
print(f"count_paths_full({m},{n}) = {count_paths_full(m,n)}")
print(f"count_paths_rolling({m},{n}) = {count_paths_rolling(m,n)}")

full_grid_bytes = sys.getsizeof([[0]*n for _ in range(m)])
rolling_bytes = sys.getsizeof([0]*n) * 2
print(f"Full grid memory: ~{full_grid_bytes} bytes")
print(f"Rolling array memory: ~{rolling_bytes} bytes")

# ── Generator vs list: O(1) vs O(n) space ────────────────────────────────────
def squares_list(n):
    return [x*x for x in range(n)]    # O(n) — full list in memory

def squares_gen(n):
    return (x*x for x in range(n))    # O(1) — one value at a time

lst = squares_list(1000)
gen = squares_gen(1000)
print(f"List of 1000 squares: {sys.getsizeof(lst)} bytes")
print(f"Generator of 1000 squares: {sys.getsizeof(gen)} bytes")
print("Space complexity demo complete.")


<a id='6'></a>

## 6. Decision Map — Reading Interview Problems

---

```
SIGNAL IN THE PROBLEM                  LIKELY COMPLEXITY TARGET
──────────────────────────────────────────────────────────────────
n ≤ 10                                 O(n!) or O(2^n) OK
n ≤ 20                                 O(2^n) OK (backtracking)
n ≤ 500                                O(n²) or O(n³) OK
n ≤ 10,000                             O(n log n) or O(n²) OK
n ≤ 10^6                               O(n log n) required
n ≤ 10^8                               O(n) required
n > 10^8                               O(log n) or O(1) required

PROBLEM SHAPE                          ALGORITHM
──────────────────────────────────────────────────────────────────
"Sort" in problem title                At least O(n log n)
"Find in sorted array"                 Binary search O(log n)
"Shortest path unweighted"             BFS O(V+E)
"Count subsets / combinations"         DP or backtracking
"Sliding window / subarray"            O(n) two-pointer
"Top K"                                Heap O(n log k)
"Two sum / pair sum"                   Hash map O(n)
"Connected components / islands"       BFS/DFS O(V+E)
"Interval overlap"                     Sort by start O(n log n)
"Next greater element"                 Monotonic stack O(n)
"LCS / edit distance"                  2D DP O(n·m)
"Coin change / climb stairs"           1D DP O(n)

OPTIMIZATION PATTERNS
──────────────────────────────────────────────────────────────────
O(n²) → O(n log n)                     Replace inner scan with binary search
O(n²) → O(n)                           Two pointers or sliding window
O(2^n) → O(n·2^n) DP with bitmask      Memoize over state
Recursive O(2^n) → O(n)               Memoization / DP
O(n·m) space → O(m) space             Rolling array DP
```


<a id='7'></a>

## 7. Cheat Sheet

---

**1. When to reach for each growth class:**

| Constraint | Max Target | Common Approach |
|------------|-----------|----------------|
| n ≤ 20 | O(2^n) | Backtracking, bitmask DP |
| n ≤ 500 | O(n²) | Double loop, 2D DP |
| n ≤ 10^4 | O(n log n) | Sort + binary search |
| n ≤ 10^6 | O(n) | Hash map, two pointer |
| n ≤ 10^8 | O(log n) | Binary search only |

**2. Built-in O(1) operations — memorize these:**

```python
# These are ALWAYS O(1):
len(anything)           # stored attribute
dict[key]               # hash lookup
set.add(x)              # hash insert (amortized)
x in dict               # hash lookup
x in set                # hash lookup
list.append(x)          # amortized O(1)
list.pop()              # pop from END only
deque.appendleft(x)     # O(1) — both ends
deque.popleft()         # O(1) — both ends

# These are NOT O(1):
list.pop(0)             # O(n) — shifts everything
list.insert(0, x)       # O(n) — shifts everything
x in list               # O(n) — linear scan
```

**3. Common templates:**

```python
# TEMPLATE: O(n) frequency count
from collections import Counter
freq = Counter(arr)           # O(n) build
top_k = freq.most_common(k)  # O(n log k)

# TEMPLATE: O(log n) binary search
lo, hi = 0, len(arr) - 1
while lo <= hi:
    mid = (lo + hi) // 2
    if arr[mid] == target: return mid
    elif arr[mid] < target: lo = mid + 1
    else: hi = mid - 1

# TEMPLATE: O(n) sliding window
left = 0
for right in range(len(arr)):
    # expand window: process arr[right]
    while window_invalid():
        # shrink window: undo arr[left]
        left += 1

# TEMPLATE: O(n log k) top-K heap
import heapq
heap = []
for x in arr:
    heapq.heappush(heap, x)
    if len(heap) > k:
        heapq.heappop(heap)   # evict smallest → keep top-k largest
```

**4. Gotchas:**

```
❌  list.pop(0) thinking it's O(1) — it's O(n)
❌  x in list thinking it's O(1) — it's O(n)
❌  string += in a loop is O(n²) total — use join()
❌  heapify called in loop is O(n log n) — build once O(n) with heapify()
✅  use deque for O(1) left-side ops
✅  use set/dict for O(1) membership
✅  heapify([...]) is O(n) — not O(n log n)
✅  rolling DP array reduces O(n·m) space to O(m)
```


```
                    📊 COMPLEXITY REFERENCE MAP

                    INPUT SIZE → ALGORITHM CHOICE

                    n > 10^8
                    └─ O(1) / O(log n)
                       Hash lookup, Binary search

                    n ≤ 10^8
                    └─ O(n)
                       Single pass, Two pointer, Sliding window

                    n ≤ 10^6
                    └─ O(n log n)
                       Sort, Heap ops, Merge sort

                    n ≤ 10^4
                    └─ O(n²)
                       Double loop, 2D DP

                    n ≤ 500
                    └─ O(n³)
                       Triple loop, Floyd-Warshall

                    n ≤ 20
                    └─ O(2^n)
                       Backtracking, Bitmask DP

                    n ≤ 12
                    └─ O(n!)
                       Full permutations

    BUILT-IN SURPRISES
    ─────────────────────────────────────────────
    list.pop(0)   → O(n)   use deque.popleft()
    x in list     → O(n)   use set membership
    str +=        → O(n²)  use "".join()
    heapify([])   → O(n)   NOT O(n log n)
    ─────────────────────────────────────────────

---
*End of Complexity Reference Guide — Sean Edition*
```
